<a href="https://colab.research.google.com/github/kuds/courtside-dynamics/blob/main/notebooks/rescore_runs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Re-score saved PaddleTennis runs under an evaluation protocol

Scores every saved policy of a set of Drive runs -- each run's best
model and every periodic checkpoint -- under one versioned evaluation
protocol, on CPU, with the in-loop evaluator's own code
(`courtside_dynamics.experiments.harness.rescore`). Nothing is
retrained and nothing is written inside a run directory: the scores
go to a separate folder beside `training_runs/`.

The default is design L3's re-scoring
(`docs/design_campaign_structure_20261016.md`, section 5 "Eras"):
the August runs, the registered run `20260816_235141` among them,
under `paddle-npoint@v3`, the PaddleTennis selection protocol with the
evaluation env's shaping stripped. Its reward is task reward only; its
k>=2 headline (`episode_rally_returns_a_ep_mean`) reads exactly what
`paddle-npoint@v2` reads, because the observations carry no escrow
state.

1. **Install** a pinned ref, then **settings**, the **protocol** and
   the **folders**.
2. **Index** the runs folder (the run index, design L1) and **pick**
   the runs.
3. **Re-score** them (re-running resumes: a run already scored for the
   same files is skipped). `QUICK = True` scores the best model and 3
   checkpoints per run, into its own `-quick` folder.
4. **Read** the per-checkpoint k>=2 and k=1 curves, **plot** them, and
   take a **`select()` verdict** per run: the checkpoint the protocol's
   selection rule picks, next to the run's own `best_model`.

A full re-score costs about 2.6 CPU-minutes per checkpoint (30 paired
1,500-step episodes on the standard block and as many on the
confirmation block), so about an hour per 2M-step run with checkpoints
every 100k; `QUICK` takes about 10 minutes per run. A CPU runtime is
enough: the protocol pins `device="cpu"`.

## 1. Install

In [ ]:
# Immutable git ref to install: a full commit SHA (or a dated tag such as
# `name-YYYYMMDD`), never a branch name. A merge to `main` must not change
# what a Colab session scores; tests/test_notebooks.py enforces the pin.
# Bump it in the commit that ships notebook changes needing newer package
# code. To smoke-test an unmerged PR, set it to that branch's head SHA.
# Placeholder: the parent of the commit that added this notebook, which
# predates paddle-npoint@v3. The week-1 wrap-up bumps it to a ref that
# ships the protocol; until then section 3 refuses to run.
REPO_REF = "0be8d77ad11a0fdfb487931bcbd3df127fe5d50d"

!pip install -q "courtside-dynamics[train,notebooks] @ git+https://github.com/kuds/courtside-dynamics@{REPO_REF}"
print(f"Installed courtside-dynamics from ref: {REPO_REF}")

## 2. Settings

Everything that changes a score lives in the protocol; these knobs only
choose runs and pace the work.

In [ ]:
USE_DRIVE = True
# The runs folder, under the Drive training-runs root
# (MyDrive/Finding Theta/courtside-dynamics/training_runs/).
RUNS_SUBDIR = "PaddleTennis/sac"
# The protocol every artifact is scored under (experiments.protocols).
PROTOCOL_ID = "paddle-npoint@v3"
# None picks every 2026-08 run with a model to score, which must include
# the registered run REGISTERED_RUN; or list run ids explicitly.
RUN_IDS = None
REGISTERED_RUN = "20260816_235141"
# The periodic series scored next to each best model: "checkpoints" (the
# August runs' checkpoints/, every 100k steps) or "snapshots" (runs
# trained with snapshot_evals keep one per evaluation).
SERIES = "checkpoints"
# QUICK scores the best model and QUICK_CHECKPOINTS of the series per run
# (spread from first to last) into a separate "-quick" folder.
QUICK = False
QUICK_CHECKPOINTS = 3
# A run whose scores already cover exactly its files under this protocol
# is skipped, so re-running resumes; FORCE re-scores it.
FORCE = False

## 3. The protocol

Refuses to go on when the installed package does not register
`PROTOCOL_ID` (bump `REPO_REF`). The fingerprint printed here is the one
every score row records.

In [ ]:
from courtside_dynamics.experiments.protocols import get_protocol, registered_protocols

if PROTOCOL_ID not in registered_protocols():
    raise ValueError(
        f"the installed package registers no {PROTOCOL_ID} (it has "
        f"{', '.join(registered_protocols())}): bump REPO_REF to a ref that ships it"
    )
PROTOCOL = get_protocol(PROTOCOL_ID)
RULE = PROTOCOL.selection_rule()
print(f"{PROTOCOL.id}  fingerprint {PROTOCOL.fingerprint}")
print(PROTOCOL.description)
print(
    "select(): "
    + ", ".join(f"{key} (delta {delta:.4g})" for key, delta in zip(RULE.keys, RULE.min_deltas))
    + (", confirmed on the second block" if RULE.confirm else "")
)

## 4. Folders

The scores go to `rescores/<runs subdir>/<protocol>/` beside
`training_runs/`, never into it: the run directories stay exactly what
`train()` left (their manifests keep verifying), and an index of
`training_runs/` never meets a score file.

In [ ]:
from pathlib import Path

from courtside_dynamics.notebook_utils import mount_drive

if USE_DRIVE:
    mount_drive()
    PROJECT_ROOT = Path("/content/drive/MyDrive/Finding Theta/courtside-dynamics")
    TRAINING_RUNS = PROJECT_ROOT / "training_runs"
else:
    PROJECT_ROOT = Path(".")
    TRAINING_RUNS = PROJECT_ROOT / "logs"  # resolve_run_dir's local root
RUNS_ROOT = TRAINING_RUNS / RUNS_SUBDIR
OUT_ROOT = PROJECT_ROOT / "rescores" / RUNS_SUBDIR / (PROTOCOL.id + ("-quick" if QUICK else ""))
if not RUNS_ROOT.is_dir():
    raise FileNotFoundError(f"no runs folder at {RUNS_ROOT}")
if OUT_ROOT.resolve().is_relative_to(TRAINING_RUNS.resolve()):
    raise ValueError(f"{OUT_ROOT} lies inside {TRAINING_RUNS}: scores never go there")
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Runs:  ", RUNS_ROOT)
print("Scores:", OUT_ROOT)

## 5. Index the runs

One row per run directory (`experiments.index`): the August runs are
backfilled from their `config.json` and `stage_summary.txt`, aborted
launches are stubs, and each row carries the campaign map's lever. The
index is written to the scores folder, as the record of what this
re-score saw. Manifests are not re-verified (`verify=False`): the
August runs have none, and re-reading every checkpoint over the Drive
mount is slow; the re-score checks every pair it scores by sha256.

In [ ]:
from courtside_dynamics.experiments.index import build_index, write_index

INDEX = build_index(RUNS_ROOT, verify=False)
ROWS = {}
for row in INDEX.rows:
    if row["run_id"] in ROWS:
        raise ValueError(f"two run directories are named {row['run_id']}; list RUN_IDS by path")
    ROWS[row["run_id"]] = row
print(f"{len(ROWS)} run(s); index: {write_index(INDEX, OUT_ROOT / 'index' / 'runs.jsonl')}")
for run_id, row in sorted(ROWS.items()):
    files = row["index"]["files"]
    lever = (row["campaign"] or {}).get("lever", "-")
    print(
        f"  {run_id}  {str(row['status']):<10} {lever:<20} "
        f"checkpoints {files['checkpoints']:>3}  best_model {'yes' if files['best_model'] else 'no '}"
        + ("  (stub)" if row["index"]["stub"] else "")
    )
for skipped in INDEX.skipped:
    print(f"  skipped {skipped.path}: {skipped.reason}")

## 6. Pick the runs

By default every 2026-08 run that is not a stub and has a best model or
a checkpoint; the registered run must be among them.

In [ ]:
def scorable(row):
    """Not a stub, and holds a best model or a checkpoint."""
    files = row["index"]["files"]
    return not row["index"]["stub"] and (files["best_model"] or files["checkpoints"] > 0)


if RUN_IDS is None:
    PICKED = [run_id for run_id, row in sorted(ROWS.items()) if run_id.startswith("202608") and scorable(row)]
    if REGISTERED_RUN not in PICKED:
        raise ValueError(f"the registered run {REGISTERED_RUN} is not a scorable run under {RUNS_ROOT}")
else:
    unknown = [run_id for run_id in RUN_IDS if run_id not in ROWS]
    if unknown:
        raise ValueError(f"no run directory named {unknown} under {RUNS_ROOT}")
    PICKED = list(RUN_IDS)
print(f"{len(PICKED)} run(s) to score: {', '.join(PICKED)}")

## 7. Re-score

Each run is scored whole or not at all: `rescore` checks every pair
(best model, checkpoint and normalizer) against the digests that bind
it before scoring anything, and a refused run is reported here with its
reasons while the others go on. Each run's rows land in
`<scores>/<run id>/scores/<protocol>.jsonl` when the run is done, so an
interrupted session loses at most the run it was on.

In [ ]:
import time

from courtside_dynamics.experiments.harness import (
    read_scores,
    rescore,
    run_artifacts,
    scores_file_name,
    series_steps,
)

WHICH = ("best", SERIES)


def quick_steps(run_dir):
    """QUICK_CHECKPOINTS steps of the run's series, spread from first to last."""
    steps = series_steps(run_dir, SERIES)
    if len(steps) <= QUICK_CHECKPOINTS:
        return steps
    last = len(steps) - 1
    spread = max(1, QUICK_CHECKPOINTS - 1)
    return sorted({steps[round(i * last / spread)] for i in range(QUICK_CHECKPOINTS)})


def already_scored(path, run_dir, steps):
    """The file holds this protocol's rows for exactly the run's files now."""
    if FORCE or not path.is_file():
        return False
    artifacts, problems = run_artifacts(run_dir, WHICH, steps=steps)
    rows = read_scores(path)
    return (
        not problems
        and all(row["protocol_fingerprint"] == PROTOCOL.fingerprint for row in rows)
        and {(row["artifact"], row["step"], row["model"]["sha256"]) for row in rows}
        == {(a.kind, a.step, a.model_sha256) for a in artifacts}
    )


SCORED, REFUSED = {}, {}
for run_id in PICKED:
    run_dir = RUNS_ROOT / ROWS[run_id]["index"]["path"]
    out_dir = OUT_ROOT / run_id
    path = out_dir / "scores" / scores_file_name(PROTOCOL)
    steps = quick_steps(run_dir) if QUICK else None
    if already_scored(path, run_dir, steps):
        print(f"{run_id}: already scored, {path}")
        SCORED[run_id] = path
        continue
    started = time.monotonic()
    try:
        [path] = rescore([run_dir], PROTOCOL, which=WHICH, out_dir=out_dir, steps=steps, device="cpu")
    except ValueError as error:
        REFUSED[run_id] = str(error)
        print(f"{run_id}: REFUSED, nothing scored. {error}")
        continue
    SCORED[run_id] = path
    print(f"{run_id}: {len(read_scores(path))} artifact(s) in {time.monotonic() - started:.0f} s, {path}")
print(f"{len(SCORED)} run(s) scored" + (f"; REFUSED: {', '.join(REFUSED)}" if REFUSED else ""))

## 8. Per-checkpoint k>=2 and k=1

The k>=2 headline is the protocol's first selection key: the policy's
confirmed returns after its first in each point, per episode. k=1 is
the policy's first confirmed return of a point, per episode
(`episode_valid_return_count_a` minus the k>=2 count). `conf` is the
same headline on the confirmation block; task reward is
`episode_reward_mean`, shaping-free under @v3.

In [ ]:
HEADLINE = RULE.keys[0]
KIND = {"checkpoints": "checkpoint", "snapshots": "snapshot"}[SERIES]


def k1(metrics):
    """k=1 conversions per episode: each point's first confirmed return."""
    return metrics["episode_valid_return_count_a_ep_mean"] - metrics["episode_rally_returns_a_ep_mean"]


SCORES = {run_id: read_scores(path) for run_id, path in SCORED.items()}
for run_id, rows in SCORES.items():
    lever = (ROWS[run_id]["campaign"] or {}).get("lever", "-")
    print(f"\n{run_id} ({lever}); seeds from {rows[0]['seed_start']} ({rows[0]['seed_source']})")
    print(f"  {'artifact':<10} {'step':>9}  {'k>=2':>6} {'conf':>6} {'k=1':>6} {'success':>7} {'task reward':>11}")
    for row in sorted(rows, key=lambda row: (row["artifact"] == "best", row["step"] or 0)):
        metrics, confirm = row["metrics"], row["confirm_metrics"] or {}
        print(
            f"  {row['artifact']:<10} {row['step']!s:>9}  {metrics[HEADLINE]:>6.3f} "
            f"{confirm.get(HEADLINE, float('nan')):>6.3f} {k1(metrics):>6.3f} "
            f"{metrics['success_rate']:>7.3f} {metrics['episode_reward_mean']:>11.3f}"
        )

## 9. Curves

One row per run: k>=2 on the left, k=1 on the right, the standard block
and the confirmation block as two lines, the run's `best_model` step as
a dashed rule. Saved as `curves.png` in the scores folder.

In [ ]:
import matplotlib.pyplot as plt

STANDARD, CONFIRM = "#2a78d6", "#eb6834"  # categorical slots 1 and 2
plotted = {run_id: rows for run_id, rows in SCORES.items() if any(r["artifact"] == KIND for r in rows)}
if plotted:
    fig, axes = plt.subplots(len(plotted), 2, figsize=(12, 2.6 * len(plotted)), squeeze=False)
    for (run_id, rows), (left, right) in zip(plotted.items(), axes):
        series = sorted((row for row in rows if row["artifact"] == KIND), key=lambda row: row["step"])
        steps = [row["step"] / 1e6 for row in series]
        best = next((row for row in rows if row["artifact"] == "best" and row["step"] is not None), None)
        for ax, read, title in ((left, lambda m: m[HEADLINE], "k>=2 per episode"), (right, k1, "k=1 per episode")):
            values = [read(row["metrics"]) for row in series]
            ax.plot(steps, values, color=STANDARD, lw=2, marker="o", ms=6, zorder=3, label="standard block")
            if all(row["confirm_metrics"] for row in series):
                confirm = [read(row["confirm_metrics"]) for row in series]
                ax.plot(steps, confirm, color=CONFIRM, lw=2, ls="--", marker="s", ms=5, zorder=2, label="confirmation block")
                values += confirm
            ax.set_ylim(0, max(0.5, 1.1 * max(values)))  # counts: the floor is 0
            if best is not None:
                ax.axvline(best["step"] / 1e6, color="0.45", lw=1, ls="--", label="best_model step")
            ax.set_title(f"{run_id}: {title}", fontsize=10)
            ax.set_xlabel("env steps (M)")
            ax.grid(color="0.85", lw=0.6)
            ax.spines[["top", "right"]].set_visible(False)
        left.legend(fontsize=8, frameon=False, loc="best")
    fig.suptitle(f"{PROTOCOL.id}, per {KIND}", y=1.0)
    fig.tight_layout()
    fig.savefig(OUT_ROOT / "curves.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved", OUT_ROOT / "curves.png")

## 10. `select()` verdicts

`experiments.select.select` applies the protocol's rule to each run's
series in step order: lexicographic on the selection keys with their
deltas, each candidate confirmed on the second block against the best's
score on that block. Its pick is set beside the run's own `best_model`
(chosen in-loop, under the protocol the run trained with) scored under
this one. With checkpoints every 100k, the August runs' series is
coarser than their 25k evaluations, so a pick here can differ from the
in-loop one for that reason alone. Saved as `verdicts.json`.

In [ ]:
from courtside_dynamics.experiments.records import write_json_atomic
from courtside_dynamics.experiments.select import select


def keys(metrics):
    return {key: metrics[key] for key in RULE.keys} if metrics else None


VERDICTS = {}
for run_id, rows in SCORES.items():
    series = [row for row in rows if row["artifact"] == KIND]
    best = next((row for row in rows if row["artifact"] == "best"), None)
    verdict = {"series": SERIES, "scored": len(series), "quick": QUICK, "select": None, "best_model": None}
    if series:
        chosen = select(series, RULE)
        verdict["select"] = {
            "step": chosen.step,
            "accepted_steps": list(chosen.accepted_steps),
            "metrics": keys(chosen.row["metrics"]),
            "confirm_metrics": keys(chosen.row["confirm_metrics"]),
            "k1": k1(chosen.row["metrics"]),
        }
    if best is not None:
        verdict["best_model"] = {"step": best["step"], "metrics": keys(best["metrics"]), "k1": k1(best["metrics"])}
    VERDICTS[run_id] = verdict
    line = f"{run_id}: "
    if verdict["select"]:
        pick = verdict["select"]
        line += (
            f"select() over {len(series)} {SERIES} picks step {pick['step']:,} "
            f"(k>=2 {pick['metrics'][HEADLINE]:.3f}, k=1 {pick['k1']:.3f})"
        )
    else:
        line += f"no {SERIES} scored"
    if verdict["best_model"]:
        own = verdict["best_model"]
        step = "?" if own["step"] is None else f"{own['step']:,}"
        line += f"; its best_model (step {step}) reads k>=2 {own['metrics'][HEADLINE]:.3f}, k=1 {own['k1']:.3f}"
    print(line)
for run_id in REFUSED:
    print(f"{run_id}: REFUSED (section 7 and verdicts.json give the reasons)")
write_json_atomic(
    OUT_ROOT / "verdicts.json",
    {
        "protocol": PROTOCOL.id,
        "protocol_fingerprint": PROTOCOL.fingerprint,
        "runs_root": str(RUNS_ROOT),
        "runs": VERDICTS,
        "refused": REFUSED,
    },
)
print("Saved", OUT_ROOT / "verdicts.json")